<a href="https://colab.research.google.com/github/HarshithReddy01/Algorithms-Practice/blob/master/moretesting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from redos_analyzer import analyze

"""This pattern looks innocent, but [\d] and [0-9] are the same."""
complex_overlap = r"(\d+|[0-9]+)*$"

print(f"Testing Hidden Overlap: {complex_overlap}")
warnings = analyze(complex_overlap)

if warnings:
    for w in warnings:
        print(f"FOUND: {w.kind}")
        print(f"DETAIL: {w.message}")
else:
    print("FAILED: Tool did not detect the overlapping character classes.")

Testing Hidden Overlap: (\d+|[0-9]+)*$
FOUND: OVERLAPPING_ALTERNATION
DETAIL: Branches '[\d]+' and '[0-9]+' share overlapping first-character sets inside quantifier '*'. The same input may be matched in multiple ways across iterations, potentially causing catastrophic backtracking.
FOUND: NESTED_QUANTIFIER
DETAIL: Repeating quantifier '+' on '[\d]' is nested inside the outer repeating quantifier.  On a failing match the engine must explore an exponential number of ways to partition the input across the two quantifiers.
FOUND: NESTED_QUANTIFIER
DETAIL: Repeating quantifier '+' on '[0-9]' is nested inside the outer repeating quantifier.  On a failing match the engine must explore an exponential number of ways to partition the input across the two quantifiers.


In [10]:
import re, time
from redos_analyzer import analyze, suggest_fix

pattern = r"([a-zA-Z0-9]+)*$"
# High-level test: 5,000 characters.
huge_input = "user_admin_access_dashboard_" * 200 + "!!!"

warnings = analyze(pattern)
if warnings:
    fix = suggest_fix(pattern, warnings[0])
    fixed_pattern = fix['fixed']

    print("Running fixed pattern against 5,000+ characters...")
    start = time.perf_counter()
    re.search(fixed_pattern + "$", huge_input)
    end = time.perf_counter()

    print(f"Fixed Pattern Time: {end - start:.6f}s")
    print("VERDICT: If time is < 0.001s, the engineering is robust.")

Running fixed pattern against 5,000+ characters...
Fixed Pattern Time: 0.000889s
VERDICT: If time is < 0.001s, the engineering is robust.


In [11]:
# Chunk 3: Nullable/Zero-Width Loop Detection
nullable_pattern = r"(a?|b?)*$"

print(f"Testing Nullable Loop: {nullable_pattern}")
warnings = analyze(nullable_pattern)

if warnings:
    for w in warnings:
        print(f"SUCCESS: Detected {w.kind}")
        print(f"EXPLANATION: {w.message}")
else:
    print("FAILED: High-level nullable loop was missed.")

Testing Nullable Loop: (a?|b?)*$
SUCCESS: Detected NULLABLE_BRANCH_IN_QUANTIFIER
EXPLANATION: Alternative(s) ['a?', 'b?'] inside quantifier '*' can match the empty string.  The group can iterate without consuming input, enabling exponential backtracking on a failing match.
